# Chapter 33: risk management

Freshly call analyze(include_french=False), save ch33_results.json and generate four new figures under RUN_DIR. The old no-write description is incorrect. Compare discrete VaR/ES, four estimators, ledger stress/liquidity/margin/limits and constructed model backtests. Loss is negative value change: positive means loss. State confidence and horizon together. Stress assumptions are conditional scenarios, not probability forecasts.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch33_risk_management_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. Shared analysis and outputs

The analysis returns structured results and arrays, then writes JSON and four constructed PNGs. French data are explicitly excluded. Chinese headings identify chapter and loss conventions.

In [ ]:
import json
import numpy as np
from quant_course.risk import empirical_var_es

ch33 = load_code_module("ch33_risk_management.py")
results, arrays = ch33.analyze(include_french=False)
ch33.generate_figures(results, arrays, RUN_DIR)
print("章节：", results["title"])
print("损失符号与单位：", results["units_and_conventions"])
(RUN_DIR / "ch33_results.json").write_text(json.dumps(results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


## 2. Discrete VaR and integrated ES

Repeat losses according to the declared scenario probabilities. Compare integrated ES with strict-tail and inclusive-tail conditional means. A mass at the quantile requires fractional allocation; neither naive conditional mean automatically equals ES.

In [ ]:
hand = results["pinned_discrete_case"]
inputs = results["teaching_inputs"]["pinned_discrete"]
counts = np.rint(np.asarray(inputs["probabilities"]) * hand["scenario_count"]).astype(int)
losses = np.repeat(inputs["loss_values"], counts)
var, es = empirical_var_es(losses, inputs["alpha"])
print("输入：", inputs)
print("VaR / ES：", var, es)
print("严格超越 VaR 的条件均值：", hand["strict_tail_conditional_mean"])
print("包含 VaR 点的条件均值：", hand["inclusive_tail_conditional_mean"])
assert np.isclose(var, hand["var"])
assert np.isclose(es, hand["es"])

## 3. Four methods on one portfolio

Compare historical, fitted normal, fitted Student-t and Monte Carlo estimates at the same horizon. The variance-matched counterexample separates tail shape from variance.

In [ ]:
comparison = results["same_portfolio_risk_methods"]
for method in ("historical", "normal_fitted", "student_t_fitted", "monte_carlo"):
    print(method, comparison[method])
print("同方差尾部反例：", comparison["variance_matched_tail_counterexample"])

## 4. Holdings, stress and limits

Trace signal/fill dates, shares, overnight gaps, asset P&L, liquidity contraction, assumed margin and actual fills after risk limits. Target weights are not filled holdings.

In [ ]:
stress = results["ledger_stress_and_limit"]
print("信号日 / 执行日：", stress["signal_date"], stress["execution_date"])
print("账本股份与隔夜情景：", stress["positions_at_signal_shares"], stress["overnight_gap_simple_returns"])
print("逐资产损益：", stress["pnl_by_asset_currency"])
print("流动性收缩：", stress["liquidity"])
print("构造保证金参数：", stress["margin"])
print("限额后的权重与实际成交：", stress["risk_limit"])

## 5. Estimation and test windows

Read Kupiec POF, zero/all-exceedance boundaries and lagged rolling quantiles. Exceedance tests depend on sampling assumptions and do not validate every tail or liquidity feature.

In [ ]:
backtest = results["risk_model_backtest"]
print("固定模型估计窗：", backtest["estimation_window"])
print("样本外检验窗与 Kupiec POF：", backtest["test_window"])
print("零次和全超越边界：", backtest["boundary_count_checks"])
print("滚动分位数时序核对：", backtest["rolling_quantile_alignment"])
print("范围：", backtest["kupiec_test"])

## 6. French branch: skipped

The current-vintage industry example is not part of analyze(False), and is not PIT or execution evidence.

In [ ]:
if RUN_FRENCH:
    real = results["french_real_example"]
    print({key: value for key, value in real.items() if key != "sha256"})
    print("SHA-256：", real["sha256"])
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


## 7. Fresh figures

Display the four PNGs written earlier in this RUN_DIR, rather than frozen French assets.

In [ ]:
from IPython.display import Image, display
for filename in ch33.FIGURES:
    display(Image(filename=str(RUN_DIR / filename)))